# 0-2 - A first classification run

This notebook does the smallest useful experiment of the whole project: it takes **10 random Ukrainian edits**, sends each one to a language model, and reads back a label.

**What you will learn**

- how to connect to the inference API and check that the connection works
- what a chat request actually contains (a system prompt with the rules, a user message with the edit)
- how to send many requests in parallel without overloading the server
- how to save results while the run is going, so nothing is lost if it crashes
- how to turn the answers into a table you can analyse

**The task given to the model.** An edit is `Potentially Manipulative` when it alters the meaning, framing, interpretation or readability of the text in a way that may be derogatory, manipulative or ideologically significant. It is `Not Problematic` when it does not meaningfully affect interpretation or framing (typos, formatting, neutral facts). It is `Ambiguous` when the change cannot be judged from text alone, for example an image swap.

**Cost note:** 10 edits means 10 API calls. Keep the number small while you are experimenting, and use a checkpoint file as soon as you scale up.

In [4]:
# Data handling and plotting.
import pandas as pd
import matplotlib.pyplot as plt

# Loading the edits (same module as in notebook 0-1).
from preprocessing import load_revisions, sample_revisions

# The classification engine and its prompt.
from classification import (
    SYSTEM_PROMPT,              # the rules sent to the model on every request
    classify_revisions,         # run a whole DataFrame through the model
    results_to_dataframe,       # turn the answers into a table
    save_results,               # write that table to a CSV
)

# Checkpointing and the API client.
from checkpoint import checkpoint_summary
from utils import MODEL, check_connection, get_client

pd.set_option('display.max_colwidth', 200)
print('model:', MODEL)

model: Qwen/Qwen3.6-35B-A3B


## 1. Connect to the API

All the models used in this project are served by one inference server. It speaks the same language as the OpenAI API, so the `openai` package works out of the box, we only have to point it at the right address.

Three things are needed:

| piece | where it comes from |
|---|---|
| the address | the constant `RCP_BASE_URL` in `utils.py` |
| the key | the environment variable `RCP_API_KEY`, or the file `.github/api_key` |
| the model name | the constant `MODEL` in `utils.py` |

`get_client()` builds the client once. Reuse it, do not create one per request.

`check_connection()` sends a single tiny request. Always run it first: it checks the key, the network and the model name in one shot, which saves you from debugging a long run that was doomed from the start.

In [5]:
# Build the client once, it will be reused for every request.
client = get_client()

# One tiny request to confirm that everything is wired correctly.
# The 'await' keyword comes from the fact that the client is asynchronous,
# see the next markdown cell for what that means.
print(await check_connection(client))

ready


## 2. Checking out the chat request

In [6]:
# The rules sent to the model with every single edit.
print(SYSTEM_PROMPT)

You are an expert Wikipedia editor analysing revision diffs for cultural manipulation.

## Task
Classify each Wikipedia edit as POTENTIALLY MANIPULATIVE, NOT PROBLEMATIC, or AMBIGUOUS.

## Decision rule
Ask yourself: does this change alter the meaning, framing, interpretation, contextual
understanding, or readability of the text in a way that is derogatory, manipulative or
ideologically significant?  Judge the EFFECT of the change, not the editor's intention.

POTENTIALLY MANIPULATIVE - the change shifts how readers understand, perceive or interpret
the subject. Examples:
- Swapping terms for ideologically loaded alternatives (Kiev to Kyiv, "annexation" to
  "reunification", "genocide" to "alleged genocide")
- Adding or removing text that introduces bias, one-sided emphasis or emotional charge
- Removing inconvenient facts or citations, as opposed to bulk cleanup
- Adding or removing quality or neutrality tags ({{POV}}, {{disputed}}) when used to
  delegitimise content
- Commenting out

## 3. Draw the sample of 10 edits

Two random draws happen here, and both are seeded:

1. `load_revisions` draws 500 edits from the whole Ukrainian file. This is the pool we work with, it is much faster than walking all 2.7 GB for ten rows.
2. `sample_revisions` draws 10 of those 500.

With `seed=42` everywhere, this notebook gives the same 10 edits on every machine and every day. That is what makes a result reproducible, and it is the first habit to build in this kind of project.

In [7]:
# A pool of 500 Ukrainian edits (about 20 seconds, the file is read once).
pool = load_revisions('Ukraine', n=500, seed=42)

# The 10 edits we are going to classify.
edits = sample_revisions(pool, n=10, seed=42)

print(f'{len(edits)} edits, from {edits["page_title"].nunique()} different articles')
edits[['idx', 'page_title', 'section', 'comment']]

10 edits, from 10 different articles


,idx,page_title,section,comment
0,400,"St Andrew's Church, Kyiv",,Dating maintenance tags: {{Use British English}}
1,33,First Secretary of the Communist Party of Ukraine,,"Added final officeholder to infobox, changed Kiev to Kyiv, added DMY dates note, fixed two names and some grammar."
2,100,Primary Chronicle,,robot Adding: [[cs:Pověst dávných let]]
3,17,Crimea,Infrastructure,/* Infrastructure */
4,153,National anthem of Ukraine,Draft lyrics,/* Draft lyrics */
5,378,Symon Petliura,,I do not think this belongs to the lead (might be included if the lead was significantly bigger)
6,213,KGB,,[[Help:Reverting|Reverted]] edits by [[Special:Contributions/72.39.116.31|72.39.116.31]] ([[User talk:72.39.116.31|talk]]) to last version by Trust Is All You Need
7,280,Ukrainian Insurgent Army,History,/* History */
8,63,Joseph Stalin,Other names,/* Other names */
9,34,Cyrillic alphabets,Summary table,/* Summary table */


## 4. Run the classification

One line of code does the whole job:

```python
results = await classify_revisions(edits, client=client, checkpoint_path=...)
```

Behind that call:

- **one task per edit**, sent in parallel. A **semaphore** limits how many requests are in flight at once (5 by default), otherwise the server would refuse the excess.
- **a deadline per request**. A request that never comes back is abandoned instead of freezing the run forever.
- **no crash on failure**. If a request fails, the result is stored with `label='Error'` and the exception message, and the other edits continue. You then decide what to do with the failures.
- **a checkpoint file**. Results are appended to a JSONL file, one line per edit, as soon as they arrive. If the notebook dies after 7 of 10 edits, running the cell again classifies the remaining 3 and reuses the 7 already stored.

The checkpoint is keyed by `idx`, so it is only valid for the same sample (same country, same `n`, same `seed`). Delete the file to force a fresh run.

Because the notebook supports `await` at the top level, no `asyncio.run` is needed. In a plain `.py` script you would write `results = asyncio.run(classify_revisions(...))`.

In [8]:
# Where the results are stored while the run is going.
# The name says which sample and which model produced them, so two runs never
# mix their records.
checkpoint_path = 'checkpoints/demo_ukraine_qwen.jsonl'

# Run the 10 edits through the model.
# verbose=True prints one line per finished edit as it arrives.
results = await classify_revisions(
    edits,
    client=client,
    checkpoint_path=checkpoint_path,
    show_progress=False,
    verbose=True,
)

# How many edits failed? A non-zero count is worth investigating before
# drawing any conclusion from the labels.
errors = [r for r in results if r['label'] == 'Error']
print(f'\n{len(results)} results, {len(errors)} errors')

Classifying 10 edits with Qwen/Qwen3.6-35B-A3B (5 at a time)
  [100] Not Problematic - The edit simply adds a standard interwiki link to the Czech Wikipedia version of...
  [153] Not Problematic - The edit adds a neutral reference note clarifying a known word-order variation i...
  [17] Not Problematic - The edit consists solely of minor formatting and grammar adjustments, such as re...
  [400] Not Problematic - The edit only adds a date parameter to a standard maintenance template, which is...
  [213] Not Problematic - The edit removes spam links and technical formatting artifacts from a bibliograp...
  [63] Not Problematic - The addition provides a straightforward, factual etymology for Stalin's adopted ...
  [34] Not Problematic - The edit merely reorders rows within a summary table, relocating entries for Ser...
  [33] Potentially Manipulative - The edit changes the toponym "Kiev" to "Kyiv" and standardizes historical figure...
  [280] Not Problematic - The edit primarily reorders 

## 5. Read the results

Each result is a small dictionary. The interesting keys are:

| key | meaning |
|---|---|
| `idx` | which edit of the sample this is, the join key with the source table |
| `label` | `Potentially Manipulative`, `Not Problematic`, `Ambiguous`, `Unknown` or `Error` |
| `reasoning` | the two sentence justification written by the model |
| `page_title`, `section`, `comment` | copied from the edit, so a result is readable on its own |
| `truncated` | `True` if the diff was cut because it was too long |
| `error` | the error message, when the call failed |

A label of `Unknown` means the model answered in a format the parser did not recognise, and `Error` means the call itself failed. Both deserve a look: they are the two ways a result can be silently useless.

In [9]:
# One result, in full, to see what the model returned.
results[0]

{'idx': 17,
 'page_title': 'Crimea',
 'section': 'Infrastructure',
 'comment': '/* Infrastructure */',
 'label': 'Not Problematic',
 'reasoning': 'The edit consists solely of minor formatting and grammar adjustments, such as removing redundant pipe displays in wiki links and standardizing list separators to en-dashes. These changes do not alter the meaning, framing, or factual content of the article.',
 'error': '',
 'truncated': False,
 'model': 'Qwen/Qwen3.6-35B-A3B'}

In [10]:
# results_to_dataframe joins the answers back to the edits they came from,
# so each row shows the label next to the page, section and comment.
results_df = results_to_dataframe(results, edits)

results_df[['idx', 'page_title', 'section', 'label', 'reasoning']]

,idx,page_title,section,label,reasoning
0,17,Crimea,Infrastructure,Not Problematic,"The edit consists solely of minor formatting and grammar adjustments, such as removing redundant pipe displays in wiki links and standardizing list separators to en-dashes. These changes do not al..."
1,33,First Secretary of the Communist Party of Ukraine,,Potentially Manipulative,"The edit changes the toponym ""Kiev"" to ""Kyiv"" and standardizes historical figures' names to Ukrainian transliterations. These toponym and naming shifts systematically impose a specific cultural an..."
2,34,Cyrillic alphabets,Summary table,Not Problematic,"The edit merely reorders rows within a summary table, relocating entries for Serbian, Montenegrin, and Abkhaz to different positions. This is a structural maintenance change that does not alter fa..."
3,63,Joseph Stalin,Other names,Not Problematic,"The addition provides a straightforward, factual etymology for Stalin's adopted surname, which is a widely documented linguistic detail. It does not introduce bias, alter the article's framing, or..."
4,100,Primary Chronicle,,Not Problematic,"The edit simply adds a standard interwiki link to the Czech Wikipedia version of the article, which is a routine maintenance and navigation improvement. It does not alter the article's content, fr..."
5,153,National anthem of Ukraine,Draft lyrics,Not Problematic,"The edit adds a neutral reference note clarifying a known word-order variation in the original Ukrainian text, which improves accuracy and readability without altering the article's meaning or fra..."
6,213,KGB,,Not Problematic,The edit removes spam links and technical formatting artifacts from a bibliography section. This is a standard maintenance cleanup that improves readability without altering the article's meaning ...
7,280,Ukrainian Insurgent Army,History,Not Problematic,"The edit primarily reorders a paragraph and adds a historical image with a factual, descriptive caption. It does not alter the narrative framing, introduce bias, or meaningfully change how readers..."
8,378,Symon Petliura,,Not Problematic,"The edit removes a detailed narrative sentence about the subject's assassination from the lead section and replaces it with early biographical details, consistent with standard Wikipedia guideline..."
9,400,"St Andrew's Church, Kyiv",,Not Problematic,"The edit only adds a date parameter to a standard maintenance template, which is a routine technical update. It does not alter the article's content, framing, or interpretation in any meaningful way."


## 6. Save the results

A result you cannot find again is a result you do not have. The two files this project writes are:

- `checkpoints/*.jsonl`, written during the run, one line per edit. This is the working file, used to resume.
- `outputs/*.csv`, written at the end, one row per edit, readable in a spreadsheet or in pandas. This is the file you share and analyse.

The CSV is written with `utf-8-sig` encoding, which adds a byte order mark. That small detail is what makes Excel open a file containing Cyrillic or Armenian text without turning it into garbage.

In [13]:
# Write the final table, with the diff included, so the file is self-contained.
path = save_results(results, 'outputs/demo_ukraine_qwen.csv', revisions=edits)
print('written to', path)

# What is in the checkpoint file? Useful to see whether a run resumed or
# started from scratch.
checkpoint_summary(checkpoint_path)

written to outputs/demo_ukraine_qwen.csv


{'path': 'checkpoints/demo_ukraine_qwen.jsonl',
 'exists': True,
 'n_results': 10,
 'n_unique_keys': 10,
 'labels': {'Not Problematic': 9, 'Potentially Manipulative': 1}}

## What next

The natural progression, in increasing order of effort:

1. **Scale up carefully.** Go from 10 to 200 edits and watch the error count. At this size a checkpoint file stops being optional.
2. **Change the model.** Edit `MODEL` in `utils.py` (or pass `model=...` to `classify_revisions`) and rerun the same 10 edits. Comparing two models on the same sample is the first experiment of the project.
3. **Compare the two countries.** Run the same 10 Ukrainian edits and 10 Armenian edits, then look at whether one context gets more `Potentially Manipulative` labels than the other. This is the bias question in its simplest form.
4. **Bring in human labels**